# Lower back nonwear: current Python vs GGIR 2023

Compare exactly three variants on the **same calibrated McRoberts recording**:

1. The legacy Python detector (`current`), unchanged.
2. GGIR 2023 with matched SD/range thresholds (currently 13 / 50 mg), requiring **at least 2 axes**.
3. An experimental GGIR 2023 variant with the **same matched thresholds**, requiring **all 3 axes**.

The three-axis requirement applies to the initial nonwear classification. Both GGIR
variants retain the same postprocessing, which can add nonwear afterwards.

This notebook is an experiment entry point, separate from the gait and TIB runners.
It reads OMX inputs and creates a new experiment directory for each run; it does not
write to existing silver/gold outputs. No temperature or DETACH is involved.

Use the project's locked Python environment and install R so `Rscript` is available.
The helper downloads three checksum-verified files from the pinned GGIR 3.3-9 release
on the first comparison run. No GGIR package installation or Python dependency upgrade
is needed. Loading previous results and plotting do not need R or a network connection.

**Start with one participant and one visit.** Full recordings can require substantial
memory because the existing reader and calibration operate at 100 Hz. Keep executed
notebook outputs private and clear them before committing this notebook.

Production now uses **GGIR 2023 matched thresholds** (two axes, SD 13 mg / range 50 mg). The saved method identifier `current` refers to the **legacy Python baseline** used when this comparison was created; it remains available for historical reproduction. Production uses a validated Python port; R is needed only to rerun this reference comparison.

In [ ]:
from pathlib import Path
import shutil
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "scripts" / "compare_lower_back_nonwear.py").is_file():
    raise RuntimeError("Open this notebook from the repository root or notebooks directory.")
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
from IPython.display import display
import importlib
import compare_lower_back_nonwear as nw

# Pick up helper fixes when reusing a notebook kernel.
nw = importlib.reload(nw)

## Private settings

The next cell creates an ignored local configuration if necessary. Edit that file
before continuing: set `bronze_root`, a dedicated `output_root`, and quoted
`participant_ids`, preserving leading zeros. An empty selection stops the comparison.
Relative paths are resolved from the configuration file's directory. There must be
exactly one OMX file in each selected participant/visit/McRoberts input folder.

In [ ]:
CONFIG_PATH = PROJECT_ROOT / "configs" / "nonwear_comparison.local.toml"
if not CONFIG_PATH.exists():
    shutil.copyfile(PROJECT_ROOT / "configs" / "nonwear_comparison.example.toml", CONFIG_PATH)
print(f"Edit your private configuration before the next cell: {CONFIG_PATH}")

In [ ]:
settings = nw.load_settings(CONFIG_PATH)
display(pd.Series({
    "participants": settings["participant_ids"],
    "visits": settings["visits"],
    "input": settings["bronze_root"],
    "experiment outputs": settings["output_root"],
    "Rscript": settings["rscript"],
}))


## Run the comparison once

This is the cell that loads the selected sensor recordings and runs all three comparison variants.
GGIR receives the calibrated acceleration in g from the same preprocessing call used
for the current Python result. Neither detector is fitted or tuned here.

Each run has a unique directory and completion manifest. A failed run cannot be
loaded as a completed comparison. The helper saves long-format `subject_summary.csv`,
per-method bouts and window decisions, one-second plot data, input metadata and GGIR
settings/source hashes. Its temporary full-resolution R input is removed afterwards.

The new three-axis variant needs a **new comparison run**; older runs do not contain
its decisions and will be rejected rather than relabelled. Existing output files are preserved.

To inspect an **existing completed run containing the three-axis variant** instead, replace the first line below with
`RUN_DIR = Path("~/dare-data/outputs/lower_back_nonwear_comparison/your_run").expanduser()`.
The optional CLI entry point is `python scripts/compare_lower_back_nonwear.py --config
configs/nonwear_comparison.local.toml`.

In [ ]:
RUN_DIR = nw.run_comparison(CONFIG_PATH)
summary = nw.load_summary(RUN_DIR)
print(f"Results: {RUN_DIR}")
display(summary)

## Load McRoberts nonwear summary

As in the reference exploration, inspect nonwear hours/percent, recording time missing
until seven days, and total unavailable time. The summary is long-format: every
participant/visit has one row per method. Missing recording time is identical across
methods and is never labelled as detected nonwear.

Histogram legends show the **median [25th–75th percentile] of all recordings**. Set
`UPPER_QUANTILE = 0.95` to hide only the upper visual tail, as in the reference notebook;
the exported values, paired comparisons and summary statistics remain unfiltered.

In [ ]:
summary = nw.load_summary(RUN_DIR)
metrics = ["detected_nonwear_hours", "detected_nonwear_percent",
           "missing_recording_hours", "missing_recording_percent",
           "total_unavailable_percent"]
display(summary.groupby("method")[metrics].agg(["median", "std"]))

UPPER_QUANTILE = 1.0
summary_figure = nw.plot_summary(summary, upper_quantile=UPPER_QUANTILE)
summary_figure


## Paired comparisons

Every point is the same participant/visit. The left panel compares current Python
with GGIR matched thresholds requiring at least two axes. The right panel directly
compares GGIR's two- and three-axis variants. Points above the diagonal have more
nonwear with the method on the y-axis. Similar total nonwear can still occur at different
times; the next table ranks sample-level disagreement rather than just differences in totals.


In [ ]:
paired_figure = nw.plot_paired(summary)
paired_figure


In [ ]:
ranked = summary.loc[summary.method.ne("current")].sort_values("disagreement_hours", ascending=False)
display(ranked[["subject", "visit", "method", "axes_required", "detected_nonwear_hours",
                "disagreement_hours", "additional_nonwear_vs_current_hours",
                "additional_wear_vs_current_hours", "internal_gap_hours",
                "unclassified_tail_minutes"]].head(20))


## Inspect acceleration and nonwear bouts

Select a participant/visit, initially the largest disagreement in the table. The three
panels have identical acceleration traces with each method's nonwear bouts shaded.
The trace is a **one-second mean with min/max envelopes from all original samples**,
not an input downsampling step. Wide views use larger display bins while retaining
the sample min/max envelope; zooming restores finer detail. Gray spans are timestamp gaps. Amber hatching marks
a final partial 15-minute block that was not classified by the detector routines.

Set `PLOT_START` and `PLOT_END` to recording-local timestamps, for example
`"2024-01-02 18:00"` and `"2024-01-03 11:00"`, to inspect a night. Plot limits only
crop the display: the detectors always receive the full selected recording, preserving
window context and recording-boundary rules. These plots show the detector wear masks,
before the production TIB-specific lying correction.

In the raw ACC/gyro cell, `RAW_NONWEAR_METHOD` selects the shaded variant:
`"ggir_2023_matched"` (at least two axes) or `"ggir_2023_matched_3axes"` (all three axes,
experimental). It initially selects the three-axis variant. Raw signals retain all
100 Hz samples; `RAW_START = None` and `RAW_END = None` show the full analysed recording.


In [ ]:
%matplotlib qt

In [ ]:
focus = ranked.iloc[0]
SUBJECT = str(focus.subject)
VISIT = str(focus.visit)
PLOT_START = None
PLOT_END = None

recording_figure = nw.plot_recording(
    RUN_DIR, SUBJECT, VISIT, start=PLOT_START, end=PLOT_END,
)
recording_figure


In [ ]:
# Raw 100 Hz ACC and gyro, shaded with GGIR matched-threshold nonwear.
# None shows the full analysed recording (up to the seven-day sample cap).
# Set either limit to a recording-local timestamp to inspect a shorter interval.
RAW_START = None
RAW_END = None

# Choose "ggir_2023_matched" for ≥2 axes, or the experimental 3-axis option:
RAW_NONWEAR_METHOD = "ggir_2023_matched_3axes"

import json
import numpy as np
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from dare_wearables.lower_back.io.mcroberts_loader import read_dp7_as_dataframe_fast

if RAW_NONWEAR_METHOD not in nw.METHODS[1:]:
    raise ValueError("Reload the helper, then select a matched two- or three-axis GGIR method.")
nw.load_summary(RUN_DIR)  # requires a completed run containing the new three-axis variant

raw_folder = Path(RUN_DIR).expanduser().resolve() / str(SUBJECT) / str(VISIT)
raw_info = json.loads((raw_folder / "recording.json").read_text())
raw_path = Path(raw_info["input"]).expanduser()

# Read the original file once; reuse it when changing the displayed interval.
if globals().get("_nw_raw_100hz_cache", {}).get("path") != raw_path:
    raw_data = read_dp7_as_dataframe_fast(raw_path, verbose=0)
    if not np.isclose(raw_data.attrs.get("sample_rate_hz", np.nan), 100.0):
        raise ValueError("This raw-signal plot requires a 100 Hz recording.")
    raw_data.set_index("datetime", inplace=True)
    raw_data.index = pd.DatetimeIndex(raw_data.index).as_unit("ns")
    _nw_raw_100hz_cache = {"path": raw_path, "data": raw_data}
raw_data = _nw_raw_100hz_cache["data"]

recording_start = pd.Timestamp(raw_info["start"])
recording_end = pd.Timestamp(raw_info["end"])
raw_left = max(recording_start, pd.Timestamp(RAW_START)) if RAW_START is not None else recording_start
raw_right = min(recording_end, pd.Timestamp(RAW_END)) if RAW_END is not None else recording_end
if raw_left >= raw_right:
    raise ValueError("Choose a raw plot interval that overlaps the analysed recording.")
raw_first = raw_data.index.searchsorted(raw_left, side="left")
raw_stop = raw_data.index.searchsorted(raw_right, side="left")
raw_view = raw_data.iloc[raw_first:raw_stop][["ax", "ay", "az", "gx", "gy", "gz"]]
if raw_view.empty:
    raise ValueError("No raw samples exist inside the selected interval.")

# Insert empty plot points at gaps, keeping all original samples and values.
# No filtering, averaging, resampling or stride-based thinning is applied.
gap_indices = np.flatnonzero(np.diff(raw_view.index.asi8) > 1.5 / 100 * 1e9)
if len(gap_indices):
    gap_midpoints = raw_view.index[gap_indices] + (
        raw_view.index[gap_indices + 1] - raw_view.index[gap_indices]
    ) / 2
    raw_view = raw_view.reindex(raw_view.index.union(gap_midpoints))

matched_bouts = pd.read_csv(
    raw_folder / f"{RAW_NONWEAR_METHOD}_bouts.csv", parse_dates=["start", "end"],
)
matched_bouts = matched_bouts.loc[
    (matched_bouts.end > raw_left) & (matched_bouts.start < raw_right)
]

raw_figure, raw_axes = plt.subplots(2, 1, figsize=(17, 8), sharex=True)
for axis, columns, ylabel, title in (
    (raw_axes[0], ("ax", "ay", "az"), "Acceleration (g)", "Raw acceleration · 100 Hz"),
    (raw_axes[1], ("gx", "gy", "gz"), "Angular velocity (°/s)", "Raw gyroscope · 100 Hz"),
):
    for column, color in zip(columns, ("#a64d79", "#26846e", "#476aa0")):
        axis.plot(raw_view.index, raw_view[column], color=color, lw=.5, label=column)
    for index, bout in enumerate(matched_bouts.itertuples(index=False)):
        axis.axvspan(
            max(raw_left, bout.start), min(raw_right, bout.end),
            color=nw.COLORS[RAW_NONWEAR_METHOD], alpha=.2,
            label=f"{nw.LABELS[RAW_NONWEAR_METHOD]} nonwear" if index == 0 else None,
        )
    axis.set(title=title, ylabel=ylabel, xlim=(raw_left, raw_right))
    axis.legend(loc="upper right", ncol=4, fontsize=9)
raw_axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%d %b\n%H:%M:%S"))
raw_axes[-1].set_xlabel("Recording-local time")
raw_figure.suptitle(f"{SUBJECT} · {VISIT} — raw ACC / gyro; shading = {nw.LABELS[RAW_NONWEAR_METHOD]} nonwear")
raw_figure.tight_layout()
raw_figure


## Initial decisions vs postprocessing

Use the same interval to see whether disagreement starts in the threshold/window stage
or is introduced by wear-gap and recording-edge relabeling. The initial score already
includes each implementation's single-axis-neighbour correction. Detailed SD/range
columns are available in the Python window CSV; GGIR's original function returns
scores rather than per-axis statistics. GGIR clipping flags are exported separately.

The three-axis variant changes the initial axis-count threshold only. Its final mask
can still include periods added by postprocessing even where fewer than three axes
met the initial criterion. Inspect the "Added" row when evaluating false sleep nonwear.


In [ ]:
decisions_figure = nw.plot_window_decisions(
    RUN_DIR, SUBJECT, VISIT, start=PLOT_START, end=PLOT_END,
)
decisions_figure


## Optional figure export

Run the next cell to save the currently displayed figures inside this experiment run.
The recording figure reflects the selected participant and plot interval.

In [ ]:
FIGURE_DIR = RUN_DIR / "figures"
FIGURE_DIR.mkdir(exist_ok=True)
for name, figure in {
    "summary": summary_figure,
    "paired": paired_figure,
    f"{SUBJECT}_{VISIT}_acceleration": recording_figure,
    f"{SUBJECT}_{VISIT}_window_decisions": decisions_figure,
}.items():
    figure.savefig(FIGURE_DIR / f"{name}.png", dpi=160, bbox_inches="tight")


## Interpretation and scope

- Both GGIR variants call the original `detect_nonwear_clipping` and `g.weardec`
  routines from **GGIR 3.3-9**, CRAN-source commit
  `5acd50f9283e02fe0efc98057978dc8e9d5629ab`. Sources and SHA-256 hashes are recorded.
  Downloaded GGIR source retains its upstream Apache-2.0 licensing; it is not vendored
  into this repository.
- This compares **detector functions**, including their original postprocessing, on
  shared calibrated samples. It does not run GGIR's sensor readers, calibration,
  protocol exclusions, imputation, or ad-hoc CSV noise-threshold logic. Both GGIR
  variants use the Python detector's matched SD/range thresholds (13 / 50 mg currently).
  Their only difference is the initial requirement of at least two or all three axes.
  Actual thresholds and `axes_required` are saved in `ggir_settings.csv`, window tables
  and the subject summary. Neither variant combines clipping with nonwear in its mask.
- Existing preprocessing retains the production cap of seven days of **samples**.
  Recorded hours and nonwear duration count observed samples at 100 Hz. Missing
  recording hours = `max(168 - observed_hours, 0)`; total unavailable percent uses
  168 hours as its denominator. Internal timestamp-gap duration is reported separately;
  do not add it again to the missing-recording estimate. This is a fixed-duration
  coverage summary, not a calendar-aligned adherence measure.
- Gaps are not filled or concatenated into extra samples. Both detector functions use
  the same observed sample sequence and sample-based windows, so a window can span
  missing elapsed time. The plots break nonwear spans at gaps and display missing time
  separately. A full GGIR pipeline can handle gaps differently.
- Both low-level routines produce complete 15-minute output epochs. Any final partial
  epoch remains outside their classification, is counted as wear for compatibility
  with the current pipeline mask, and is explicitly reported/shaded as unclassified.
- The current baseline preserves the production wear mask, including its microsecond
  bout boundaries against nanosecond sensor timestamps. Diagnostic epochs use their
  original sample-count bounds. `boundary_rounding_difference_samples` reports any
  difference between those epoch labels and the applied mask; the consistency check
  reconstructs merged bouts exactly as production does.
- More or less detected nonwear is not evidence of greater accuracy. Inspect suspicious
  intervals alongside removal logs or other independent evidence before changing defaults.

Upstream references: [GGIR parameter source](https://github.com/cran/GGIR/blob/5acd50f9283e02fe0efc98057978dc8e9d5629ab/R/load_params.R),
[detector](https://github.com/cran/GGIR/blob/5acd50f9283e02fe0efc98057978dc8e9d5629ab/R/detect_nonwear_clipping.R),
[postprocessing](https://github.com/cran/GGIR/blob/5acd50f9283e02fe0efc98057978dc8e9d5629ab/R/g.weardec.R).